In [ ]:
import pandas as pd
import json
from pprint import pprint

In [ ]:
#region = "us-east-2"
region = "us-east-1"

This code reads the file generated by the following command:
```
aws ec2 describe-instances     --filters Name=instance-state-name,Values=running,stopped \
    --region ${region} \
    --query 'Reservations[*].Instances[*].{Name:Tags[?Key==`Name`]|[0].Value,Instance:InstanceId,Type:InstanceType,State:State.Name,SecurityGroups:SecurityGroups}' \
    --output text > instances.csv

aws elbv2 describe-load-balancers --query 'LoadBalancers[*].{Name:LoadBalancerName,SecurityGroups:SecurityGroups}' --region ${region} --output text > albs.csv

aws rds describe-db-instances --region ${region} --query 'DBInstances[*].{Name:DBInstanceIdentifier,Engine:Engine,SecurityGroups:VpcSecurityGroups[*].{Name:VpcSecurityGroupId}}' > rds.json
```

In [ ]:
%%bash
aws ec2 describe-instances     --filters Name=instance-state-name,Values=running,stopped \
    --region us-east-2 \
    --query 'Reservations[*].Instances[*].{Name:Tags[?Key==`Name`]|[0].Value,Instance:InstanceId,Type:InstanceType,State:State.Name,SecurityGroups:SecurityGroups}' \
    --output text > /tmp/instances.csv
aws elbv2 describe-load-balancers --query 'LoadBalancers[*].{Name:LoadBalancerName,SecurityGroups:SecurityGroups}' --region us-east-2 --output text > /tmp/albs.csv
aws rds describe-db-instances --region us-east-2 --query 'DBInstances[*].{Name:DBInstanceIdentifier,Engine:Engine,SecurityGroups:VpcSecurityGroups[*].{Name:VpcSecurityGroupId}}' > /tmp/rds.json

In [ ]:
%%bash
aws ec2 describe-instances     --filters Name=instance-state-name,Values=running,stopped \
    --query 'Reservations[*].Instances[*].{Name:Tags[?Key==`Name`]|[0].Value,Instance:InstanceId,Type:InstanceType,State:State.Name,SecurityGroups:SecurityGroups}' \
    --output text > /tmp/instances.csv
aws elbv2 describe-load-balancers --query 'LoadBalancers[*].{Name:LoadBalancerName,SecurityGroups:SecurityGroups}' --output text > /tmp/albs.csv
aws rds describe-db-instances --query 'DBInstances[*].{Name:DBInstanceIdentifier,Engine:Engine,SecurityGroups:VpcSecurityGroups[*].{Name:VpcSecurityGroupId}}' > /tmp/rds.json

In [ ]:
inst_fname = "/tmp/instances.csv"
alb_fname = "/tmp/albs.csv"
rds_fname = "/tmp/rds.json"

In [ ]:
instance_l = []
sg_l = []
inst_sg_l = None
most_recent_inst = None
with open(inst_fname) as f:
    for idx, line in enumerate(f):
        words = line.split("\t")
        words = [word.strip() for word in words]
        if words[0] == "SECURITYGROUPS":
            assert inst_sg_l is not None
            assert most_recent_inst is not None
            inst_sg_l.append({"name":words[1], "descr":words[2]})
        else:
            if inst_sg_l:
                assert most_recent_inst is not None
                most_recent_inst["sg"] = inst_sg_l
                sg_l.extend(inst_sg_l)
            inst_sg_l = []
            if most_recent_inst:
                instance_l.append(most_recent_inst)
            most_recent_inst = {"name":words[0], "descr":words[1], "state":words[2], "type":words[3]}
    assert most_recent_inst is not None
    most_recent_inst["sg"] = inst_sg_l
    sg_l.extend(inst_sg_l)
    instance_l.append(most_recent_inst)
print("SECURITY GROUPS FOLLOW")
pprint(sg_l)
print("INSTANCES FOLLOW")
pprint(instance_l)

In [ ]:
alb_l = []
alb_sg_l = None
most_recent_alb = None
with open(alb_fname) as f:
    for idx, line in enumerate(f):
        words = line.split("\t")
        words = [word.strip() for word in words]
        if words[0] == "SECURITYGROUPS":
            assert alb_sg_l is not None
            assert most_recent_alb is not None
            alb_sg_l.append({"name":words[1]})
        else:
            if alb_sg_l:
                assert most_recent_alb is not None
                most_recent_alb["sg"] = alb_sg_l
                sg_l.extend(alb_sg_l)
            alb_sg_l = []
            if most_recent_alb:
                alb_l.append(most_recent_alb)
            most_recent_alb = {"name":words[0]}
    assert most_recent_alb is not None
    most_recent_alb["sg"] = alb_sg_l
    sg_l.extend(alb_sg_l)
    alb_l.append(most_recent_alb)
print("SECURITY GROUPS FOLLOW")
pprint(sg_l)
print("ALBS FOLLOW")
pprint(alb_l)

In [ ]:
rds_l = []
rds_sg_l = None
with open(rds_fname) as f:
    rds_json = json.load(f)
    for elt in rds_json:
        pprint(elt["SecurityGroups"])
        elt_d = {
            "name": elt["Name"],
            "engine": elt["Engine"],
            "sg": [{"name":eltelt["Name"]} for eltelt in elt["SecurityGroups"]]  
        }
        rds_l.append(elt_d)
        sg_l.extend(elt_d["sg"])
print("SECURITY GROUPS FOLLOW")
pprint(sg_l)
print("RDS's FOLLOW")
pprint(rds_l)

In [ ]:
# Make the security groups unique
sg_name_set = set(sg["name"] for sg in sg_l)
sg_d = {sg["name"]: sg for sg in sg_l if "descr" in sg}
for sg_name in sg_name_set:
    if sg_name not in sg_d:
        sg_d[sg_name] = {"name": sg_name, "descr": "None"}
sg_l = sg_d.values()

In [ ]:
def draw_instance(inst, dotfile):
    if inst["state"] == "running":
        clr = "green"
    elif inst["state"] == "stopped":
        clr = "blue"
    else:
        clr = "red"
    if inst["descr"] != "None":
        lblstr = f"{inst['descr']}\n{inst['name']}"
    else:
        lblstr = f"--noname--\n{inst['name']}"
    dotfile.write(f"\"{inst['name']}\"[shape=rectangle, color={clr}, label=\"{lblstr}\"];\n")

def draw_alb(alb, dotfile):
    clr = "black"
    lblstr = f"{alb['name']}"
    dotfile.write(f"\"{alb['name']}\"[shape=diamond, color={clr}, label=\"{lblstr}\"];\n")

def draw_rds(rds, dotfile):
    clr = "black"
    lblstr = f"{rds['name']}\n{rds['engine']}"
    dotfile.write(f"\"{rds['name']}\"[shape=cylinder, color={clr}, label=\"{lblstr}\"];\n")

def draw_sg(sg, dotfile):
    if sg["descr"] != "None":
        lblstr = f"{sg['descr']}\n{sg['name']}"
    else:
        lblstr = f"--noname--\n{sg['name']}"
    dotfile.write(f"\"{sg['name']}\"[shape=oval, label=\"{lblstr}\"];\n")

with open("/tmp/aws_map.dot", "w") as dotfile:
    dotfile.write("graph {\n")

    dotfile.write("subgraph cluster_solr {\n")
    dotfile.write("label=\"PROD SOLR\";\n")
    for inst in instance_l:
        if "solr-0323" in inst.get("descr" ""):
            draw_instance(inst, dotfile)
    for sg in sg_l:
        if "solr" in sg["name"].lower():
            draw_sg(sg, dotfile)
    dotfile.write("};\n")
    
    for inst in instance_l:
        if "solr-0323" not in inst.get("descr", ""):
            draw_instance(inst, dotfile)    
        
    for alb in alb_l:
        draw_alb(alb, dotfile)
        
    for rds in rds_l:
        draw_rds(rds, dotfile)

    for sg in sg_l:
        if "solr" not in sg["name"].lower():
            draw_sg(sg, dotfile)

    for inst in instance_l:
        for sg in inst["sg"]:
            dotfile.write(f"\"{inst['name']}\" -- \"{sg['name']}\";\n")
            
    for alb in alb_l:
        for sg in alb["sg"]:
            dotfile.write(f"\"{alb['name']}\" -- \"{sg['name']}\";\n")

    for rds in rds_l:
        for sg in rds["sg"]:
            dotfile.write(f"\"{rds['name']}\" -- \"{sg['name']}\";\n")

    dotfile.write("}\n")

A good way to render the resulting .dot file is with:
```
neato -Tsvg -oaws_map.svg -Goverlap=false aws_map.dot
```

In [ ]:
%%bash
neato -Tsvg -o/tmp/aws_map.svg -Goverlap=false /tmp/aws_map.dot